In [ ]:
from sklearn.linear_model import LogisticRegression

In [6]:
logreg_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced',
    random_state=42
)
print("Logistic Regression Model Instatiated: ", logreg_model)

Logistic Regression Model Instatiated:  LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)


In [9]:
import time
from pathlib import Path
import sys

In [10]:
NOTEBOOK_DIR = Path.cwd()
MODEL_TRAINING_DIR = NOTEBOOK_DIR.parent
sys.path.append(str(MODEL_TRAINING_DIR))

In [12]:
from preprocessing import load_shoppers_data, preprocess_and_split
raw_df = load_shoppers_data()
x_train_processed, x_test_processed, y_train, y_test = preprocess_and_split(raw_df)

[INFO] Loading dataset from local cache: c:\Users\l\OneDrive\Desktop\wohoo\AlgoArena\model_training\data\raw\online_shoppers_intention.csv
[INFO] Saved fitted scaler to c:\Users\l\OneDrive\Desktop\wohoo\AlgoArena\model_training\artifacts\scaler.pkl
[INFO] Saved fitted encoder to c:\Users\l\OneDrive\Desktop\wohoo\AlgoArena\model_training\artifacts\encoder.pkl


In [14]:
start_train = time.perf_counter()
logreg_model.fit(x_train_processed, y_train)
train_time_sec = time.perf_counter() - start_train

In [15]:
y_pred_logreg = logreg_model.predict(x_test_processed)
y_prob_logreg = logreg_model.predict_proba(x_test_processed)[:, 1]

In [16]:
print(f"Training completed in {train_time_sec: .4f} seconds")
print(f"First 5 predictions: {y_pred_logreg[:5]}")
print(f"First 5 purchase probabilities: {y_prob_logreg[:5].round(4)}")

Training completed in  0.0637 seconds
First 5 predictions: [0 0 1 0 0]
First 5 purchase probabilities: [0.0678 0.2424 0.9999 0.0553 0.0495]


In [17]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

In [18]:
acc = accuracy_score(y_test, y_pred_logreg)
prec = precision_score(y_test, y_pred_logreg, pos_label=1)
rec = recall_score(y_test, y_pred_logreg, pos_label=1)
f1 = f1_score(y_test, y_pred_logreg, pos_label=1)

print("===LOGISTIC REGRESSION EVALUATION===")
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1-score: {f1:.4f}\n")
print(classification_report(y_test, y_pred_logreg, target_names=['No Purchase (0)', 'Purchase (1)']))

===LOGISTIC REGRESSION EVALUATION===
Accuracy: 0.8500
Precision: 0.5107
Recall: 0.7487
F1-score: 0.6072

                 precision    recall  f1-score   support

No Purchase (0)       0.95      0.87      0.91      2084
   Purchase (1)       0.51      0.75      0.61       382

       accuracy                           0.85      2466
      macro avg       0.73      0.81      0.76      2466
   weighted avg       0.88      0.85      0.86      2466



In [19]:
import numpy as np

In [20]:
for threshold in np.arange(0.3, 0.7, 0.05):
    y_pred_custom = (y_prob_logreg >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred_custom)
    prec = precision_score(y_test, y_pred_custom)
    rec = recall_score(y_test, y_pred_custom)
    print(f"Threshold: {threshold:.2f} | Precision: {prec:.2f} | Recall: {rec:.4f} | F1: {f1:.4f}")

Threshold: 0.30 | Precision: 0.33 | Recall: 0.9188 | F1: 0.4835
Threshold: 0.35 | Precision: 0.37 | Recall: 0.8822 | F1: 0.5177
Threshold: 0.40 | Precision: 0.41 | Recall: 0.8351 | F1: 0.5481
Threshold: 0.45 | Precision: 0.45 | Recall: 0.7880 | F1: 0.5761
Threshold: 0.50 | Precision: 0.51 | Recall: 0.7487 | F1: 0.6072
Threshold: 0.55 | Precision: 0.57 | Recall: 0.7016 | F1: 0.6276
Threshold: 0.60 | Precision: 0.60 | Recall: 0.6518 | F1: 0.6248
Threshold: 0.65 | Precision: 0.62 | Recall: 0.5995 | F1: 0.6099


In [21]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'solver': ['lbfgs', 'saga']
}

grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    param_grid=param_grid,
    scoring='f1', # Optimizing directly for F1-score
    cv=5
)

grid_search.fit(x_train_processed, y_train)
print("Best C:", grid_search.best_params_)
print("Best Cross-Validation F1:", grid_search.best_score_)

c:\Users\l\OneDrive\Desktop\wohoo\AlgoArena\model_training\venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Best C: {'C': 0.01, 'solver': 'saga'}
Best Cross-Validation F1: 0.6337167682019722


In [24]:
sample_row = x_test_processed.iloc[[0]].values

for _ in range(100):
    _ = logreg_model.predict(sample_row)

latencies_ms = []
for _ in range(1000):
    start = time.perf_counter()
    _ = logreg_model.predict(sample_row)
    end = time.perf_counter()
    latencies_ms.append((end-start)*1000.0)

avg_latency_logreg = float(np.mean(latencies_ms))
std_latency_logreg = float(np.std(latencies_ms))
print(f"Logistic Regression Single-Row Latency: {avg_latency_logreg:.4f} ms")

Logistic Regression Single-Row Latency: 0.0990 ms


c:\Users\l\OneDrive\Desktop\wohoo\AlgoArena\model_training\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\l\OneDrive\Desktop\wohoo\AlgoArena\model_training\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\l\OneDrive\Desktop\wohoo\AlgoArena\model_training\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\l\OneDrive\Desktop\wohoo\AlgoArena\model_training\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\l\OneDrive\Desktop\wohoo\AlgoArena\model_traini

In [28]:
import os, joblib
from pathlib import Path

In [29]:
LOGISTIC_REGRESSION_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(LOGISTIC_REGRESSION_DIR)

# Constants & Paths
ARTIFACTS_DIR = os.path.join(LOGISTIC_REGRESSION_DIR, "artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
model_path = os.path.join(ARTIFACTS_DIR, "logistic_regression.pkl")
joblib.dump(logreg_model, model_path)
file_size_kb = os.path.getsize(model_path) / 1024.0
print(f"Serialized model saved to: {model_path}")
print(f"Model file size: {file_size_kb:.2f} KB")

NameError: name '__file__' is not defined